## MCP Servers with Claude Managed Agents

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Creating the Anthropic Client

In [ ]:
import anthropic 

client = anthropic.Anthropic(api_key = claude_api_key)

### Create the Agent

In [ ]:
agent = client.beta.agents.create(
    name="Demo-MCP-Agent",
    model=claude_model_name,
    mcp_servers=[
        {
            "type": "url",
            "name": "Microsoft Learn",
            "url": "https://learn.microsoft.com/api/mcp",
        },
    ],
    system="You are a helpful AI Assistant.",
    tools=[
        {"type": "agent_toolset_20260401"},
        {"type": "mcp_toolset",  "mcp_server_name": "Microsoft Learn", "default_config": {"enabled": True}}
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### Create an Environment

In [ ]:
environment = client.beta.environments.create(
    name="quickstart-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

### Start a Session

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Quickstart session",
)

print(f"Session ID: {session.id}")

### Execute the Agent

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": """Provide Information using the MS Learn MCP Server - What is the Microsoft AI-103 Certification?
                                           Provide appropriate links to MS Learn Webpages wherever possible""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break